# Traitement des ficheirs BB_RR pour avoir les intervalles RR integrable dans Kubios

# Path à changer en fonction du fichier et enregistrer dans un fichier fixe

In [11]:
import os
import re
import pandas as pd
import numpy as np

def nettoyer_bb_rr_pour_kubios(
    input_path,
    output_dir=r"C:\Users\judupont\Desktop\BB_RR_pourkubios",
    timestamp_col="Timestamp",
    rtor_col="RtoR",
    suffix="_KUBIOS"
):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(input_path)

    rr_ms = []
    timestamps = []

    val_prec = None
    signe_prec = None
    ts_premiere = None

    for _, row in df.iterrows():
        val = float(row[rtor_col])
        ts = row[timestamp_col]

        if val == 0:
            continue

        signe = 1 if val >= 0 else -1

        if signe_prec is not None and signe != signe_prec:
            rr_ms.append(abs(val_prec) * 1000)
            timestamps.append(ts_premiere)
            ts_premiere = ts
        elif signe_prec is None:
            ts_premiere = ts

        val_prec = val
        signe_prec = signe

    if val_prec is not None and ts_premiere is not None:
        rr_ms.append(abs(val_prec) * 1000)
        timestamps.append(ts_premiere)

    df_out = pd.DataFrame({
        "Timestamp": pd.to_datetime(
            timestamps,
            format="%d/%m/%Y %H:%M:%S.%f",
            dayfirst=True
        ),
        "RR_ms": np.round(rr_ms).astype(int)
    })

    # 🔹 Extraction ID patient
    match = re.search(r"\d{4}[A-Z]{3}", input_path)
    patient_id = match.group(0) if match else "UNKNOWN"

    base_name = os.path.basename(input_path)
    name, _ = os.path.splitext(base_name)

    output_path = os.path.join(
        output_dir,
        f"{patient_id}_{name}{suffix}.csv"
    )

    df_out.to_csv(output_path, index=False)

    print("✔ Fichier Kubios prêt")
    print("  →", output_path)
    print("✔ RR extraits :", len(df_out))

    return df_out

In [12]:
input_file = r"C:\Users\judupont\Desktop\bb_rr_tronqués_v2\offset_0101EMS_2019_01_22-10_10_43_BB_RR.csv"
df_kubios = nettoyer_bb_rr_pour_kubios(input_file)

✔ Fichier Kubios prêt
  → C:\Users\judupont\Desktop\BB_RR_pourkubios\0101EMS_offset_0101EMS_2019_01_22-10_10_43_BB_RR_KUBIOS.csv
✔ RR extraits : 8697


# Bloc par bloc 

In [13]:
import os
import re
import pandas as pd
import numpy as np

def nettoyer_bb_rr_pour_kubios(
    input_path,
    output_dir=r"C:\Users\judupont\Desktop\BB_RR_pourkubios",
    timestamp_col="Timestamp",
    rtor_col="RtoR",
    suffix="_KUBIOS"
):

    os.makedirs(output_dir, exist_ok=True)

    # 🔹 Lecture fichier
    df = pd.read_csv(input_path)

    rr_ms = []
    timestamps = []

    val_prec = None
    signe_prec = None
    ts_premiere = None

    for _, row in df.iterrows():
        val = float(row[rtor_col])
        ts = row[timestamp_col]

        if val == 0:
            continue

        signe = 1 if val >= 0 else -1

        if signe_prec is not None and signe != signe_prec:
            rr_ms.append(abs(val_prec) * 1000)
            timestamps.append(ts_premiere)
            ts_premiere = ts
        elif signe_prec is None:
            ts_premiere = ts

        val_prec = val
        signe_prec = signe

    if val_prec is not None and ts_premiere is not None:
        rr_ms.append(abs(val_prec) * 1000)
        timestamps.append(ts_premiere)

    df_out = pd.DataFrame({
        "Timestamp": pd.to_datetime(
            timestamps,
            errors="coerce"
        ),
        "RR_ms": np.round(rr_ms).astype(int)
    })

    # 🔹 Extraction nom fichier uniquement
    base_name = os.path.basename(input_path)
    name, _ = os.path.splitext(base_name)

    # 🔹 Extraction ID patient + bloc
    match = re.match(r"(\d{4}[A-Z]{3})_(bloc\d+)", name, re.IGNORECASE)

    if match:
        patient_id = match.group(1)
        bloc = match.group(2)
        new_name = f"{patient_id}_{bloc}{suffix}.csv"
    else:
        new_name = f"{name}{suffix}.csv"

    output_path = os.path.join(output_dir, new_name)

    df_out.to_csv(output_path, index=False)

    print("✔ Fichier Kubios prêt")
    print("  →", output_path)
    print("✔ RR extraits :", len(df_out))

    return df_out

In [15]:
input_file = r"C:\Users\judupont\Desktop\bb_rr_tronque_blocs_v2\0102PCR_bloc1.csv"
df_kubios = nettoyer_bb_rr_pour_kubios(input_file)

✔ Fichier Kubios prêt
  → C:\Users\judupont\Desktop\BB_RR_pourkubios\0102PCR_bloc1_KUBIOS.csv
✔ RR extraits : 1678


# Tous les fichiers

In [22]:
import os
import re
import pandas as pd
import numpy as np

# ======================================================
# TRAITEMENT DES FICHIER BB_RR
# ======================================================
def nettoyer_bb_rr_pour_kubios(input_path, timestamp_col="Timestamp", rtor_col="RtoR"):
    """
    Nettoie un fichier BB_RR pour Kubios : convertit RtoR en RR_ms (temps entre deux pics R). 
    Suppression de la colonne de temps, de la colonne BR (breath rate), des répetitions dans la colonne RtoR, 
    conservation de la première valeur lors d'un changmenent de signe 
    
    Fichier BB_RR type : 
    Timestamp, BR, RotR
    2019-04-04 08:34:38.407,1642,0.0
    2019-04-04 08:34:38.463,1641,0.0
                .
                .
                .
    2019-04-04 08:34:44.511,1636,-9.41
                .
                .
                .
    2019-04-04 08:34:45.743,1637,1.114
    2019-04-04 08:34:45.799,1637,1.114
                .
                .
                .

    Fichier RR type après traitement:
    Timestamp,RR_ms
    ,0
    ,9.41
    ,1.114
    .
    .
    .
    """
    df = pd.read_csv(input_path)

    rr_ms = []
    timestamps = []

    val_prec = None
    signe_prec = None
    ts_premiere = None

    for _, row in df.iterrows():
        val = float(row[rtor_col])
        ts = row[timestamp_col]

        if val == 0:
            continue

        signe = 1 if val >= 0 else -1

        if signe_prec is not None and signe != signe_prec:
            rr_ms.append(abs(val_prec) * 1000)
            timestamps.append(ts_premiere)
            ts_premiere = ts
        elif signe_prec is None:
            ts_premiere = ts

        val_prec = val
        signe_prec = signe

    if val_prec is not None and ts_premiere is not None:
        rr_ms.append(abs(val_prec) * 1000)
        timestamps.append(ts_premiere)

    df_out = pd.DataFrame({
        "Timestamp": pd.to_datetime(
            timestamps,
            format="%d/%m/%Y %H:%M:%S.%f",
            dayfirst=True,
            errors="coerce"
        ),
        "RR_ms": np.round(rr_ms).astype(int)
    })

    return df_out

# =====================================================
# TRAITEMENT D'UN DOSSIER COMPLET AVEC FILTRE RR > 400 
# =====================================================

def traiter_dossier_bb_rr(input_dir):
    input_dir = os.path.abspath(input_dir)

    # Pour v2 et v4 (les fichiers entrant doivent faire apparaitre le numéro de la visite)
    if "v2" in input_dir.lower():
        output_dir = os.path.join(os.path.dirname(input_dir), "bb_rr_bloc_v2_kubios")
    elif "v4" in input_dir.lower():
        output_dir = os.path.join(os.path.dirname(input_dir), "bb_rr_bloc_v4_kubios")
    else:
        raise ValueError("Impossible de détecter v2 ou v4 dans le nom du dossier")

    os.makedirs(output_dir, exist_ok=True)

    fichiers = sorted(f for f in os.listdir(input_dir) if f.lower().endswith(".csv"))
    print(f"\n📂 {len(fichiers)} fichiers détectés dans : {input_dir}")
    print(f"📁 Sortie → {output_dir}\n")

    # --- regrouper par PID ---
    fichiers_par_pid = {}
    for f in fichiers:
        match = re.search(r"\d{4}[A-Z]{3}", f.upper())
        if match:
            pid = match.group(0)
            fichiers_par_pid.setdefault(pid, []).append(f)

    # --- compteur total fichiers sauvegardés ---
    total_sauvegardes = 0

    # --- traitement par candidat ---
    for pid, fichiers_pid in fichiers_par_pid.items():
        blocs_valides = []

        for fichier in fichiers_pid:
            chemin = os.path.join(input_dir, fichier)
            try:
                df_out = nettoyer_bb_rr_pour_kubios(chemin)
            except Exception as e:
                print(f"❌ Erreur {fichier} : {e}")
                continue

            if len(df_out) >= 400:
                blocs_valides.append((fichier, df_out))
            else:
                print(f"⚠️ {pid} | {fichier} supprimé (RR < 400)")

        # --- si tous les blocs sont OK, on les enregistre ---
        if len(blocs_valides) == len(fichiers_pid):
            for fichier, df_out in blocs_valides:
                # garder **nom original** sans suffixe
                chemin_sortie = os.path.join(output_dir, fichier)
                df_out.to_csv(chemin_sortie, index=False)
                total_sauvegardes += 1
                print(f"✔ {fichier} sauvegardé ({len(df_out)} RR)")
        else:
            print(f"⚠️ {pid} | Bloc(s) supprimé(s), tous les blocs du candidat ignorés")

    print(f"\n🎯 Traitement terminé | Fichiers conservés : {total_sauvegardes}")

In [20]:
traiter_dossier_bb_rr(r"C:\Users\judupont\Desktop\bb_rr_tronque_blocs_v2")



📂 453 fichiers détectés dans : C:\Users\judupont\Desktop\bb_rr_tronque_blocs_v2
📁 Sortie → C:\Users\judupont\Desktop\bb_rr_bloc_v2_kubios

✔ 0101CAR_bloc1.csv sauvegardé (1524 RR)
✔ 0101CAR_bloc2.csv sauvegardé (3813 RR)
✔ 0101CAR_bloc3.csv sauvegardé (3575 RR)
✔ 0101EMS_bloc1.csv sauvegardé (1858 RR)
✔ 0101EMS_bloc2.csv sauvegardé (3170 RR)
✔ 0101EMS_bloc3.csv sauvegardé (3494 RR)
✔ 0102PCR_bloc1.csv sauvegardé (1678 RR)
✔ 0102PCR_bloc2.csv sauvegardé (1514 RR)
✔ 0102PCR_bloc3.csv sauvegardé (940 RR)
✔ 0103BPS_bloc1.csv sauvegardé (693 RR)
✔ 0103BPS_bloc2.csv sauvegardé (2055 RR)
✔ 0103BPS_bloc3.csv sauvegardé (2324 RR)
✔ 0103SHS_bloc1.csv sauvegardé (723 RR)
✔ 0103SHS_bloc2.csv sauvegardé (2765 RR)
✔ 0103SHS_bloc3.csv sauvegardé (3057 RR)
⚠️ 0104FJS | 0104FJS_bloc1.csv supprimé (RR < 400)
⚠️ 0104FJS | 0104FJS_bloc2.csv supprimé (RR < 400)
⚠️ 0104FJS | 0104FJS_bloc3.csv supprimé (RR < 400)
⚠️ 0104FJS | Bloc(s) supprimé(s), tous les blocs du candidat ignorés
✔ 0104IBS_bloc1.csv sauveg

In [23]:
traiter_dossier_bb_rr( r"C:\Users\judupont\Desktop\bb_rr_tronque_blocs_v4")


📂 402 fichiers détectés dans : C:\Users\judupont\Desktop\bb_rr_tronque_blocs_v4
📁 Sortie → C:\Users\judupont\Desktop\bb_rr_bloc_v4_kubios

⚠️ 0101CAR | 0101CAR_bloc1.csv supprimé (RR < 400)
⚠️ 0101CAR | 0101CAR_bloc2.csv supprimé (RR < 400)
⚠️ 0101CAR | 0101CAR_bloc3.csv supprimé (RR < 400)
⚠️ 0101CAR | Bloc(s) supprimé(s), tous les blocs du candidat ignorés
✔ 0101EMS_bloc1.csv sauvegardé (3303 RR)
✔ 0101EMS_bloc2.csv sauvegardé (2925 RR)
✔ 0101EMS_bloc3.csv sauvegardé (3201 RR)
✔ 0102EMS_bloc1.csv sauvegardé (640 RR)
✔ 0102EMS_bloc2.csv sauvegardé (1845 RR)
✔ 0102EMS_bloc3.csv sauvegardé (2705 RR)
✔ 0102PCR_bloc1.csv sauvegardé (1003 RR)
✔ 0102PCR_bloc2.csv sauvegardé (3683 RR)
✔ 0102PCR_bloc3.csv sauvegardé (1735 RR)
✔ 0103BPS_bloc1.csv sauvegardé (1748 RR)
✔ 0103BPS_bloc2.csv sauvegardé (2046 RR)
✔ 0103BPS_bloc3.csv sauvegardé (2293 RR)
⚠️ 0104FJS | 0104FJS_bloc1.csv supprimé (RR < 400)
⚠️ 0104FJS | 0104FJS_bloc2.csv supprimé (RR < 400)
⚠️ 0104FJS | 0104FJS_bloc3.csv supprimé (RR <